# DataFrame.show：按列配置表格格式

`show()` 与 `style_bin_table()` 支持以下与 `dataframe2excel()` 同名的参数。

| 参数 | 用途 |
|---|---|
| `percent_cols` | 百分比列；结合 `precision` 设置显示小数位数 |
| `condition_cols` | 柱状数据条列 |
| `color_cols` | 颜色块渐变列 |
| `condition_color` | 单色、2/3 色列表或按列配置的字典；颜色为六位十六进制，可带 `#` |
| `theme_color` | 表头主题色，同时影响隔行底色和 hover |
| `fill` | `True` 显示隔行底色（默认），`False` 使用白底；不影响显式条件格式与 hover |

**优先级**：`None` 沿用自动识别；显式列名单替换该类自动规则，`[]` 关闭该类自动格式。显式百分比列优先于 `percent_format`，显式数据条列优先于 `highlight_*`。同列重叠时 `color_cols` 优先于数据条。显式指定的列在 `compact=True` 时仍保留。

多级表头可用完整 tuple 精确选择，或末级名称批量选择。颜色字典优先匹配完整 tuple，其次由内向外匹配层级名。数据条使用色阶末色；双色为最小值到最大值，三色以中位数为中值，单色以零为白色基准。复杂值仅按文本显示，不参与条件着色。

In [1]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd() if (Path.cwd() / 'hscredit').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from hscredit import OptimalBinning
from hscredit.report.feature_analyzer import benchmark_binning_methods

# 使用真实放款样本，不修改共享环境或原始数据。
df = pd.read_excel(root / 'examples' / 'hscredit_yyp.xlsx')
binner = OptimalBinning(method='quantile', max_n_bins=5, n_jobs=1)
binner.fit(df[['衡枢鉴真分老客版']], df['FPD'])
table = binner.get_bin_table('衡枢鉴真分老客版')
original = table.copy(deep=True)

## 1. 百分比、数据条、颜色块和逐列配色

In [2]:
table.show(
    percent_cols=['样本占比', '坏样本率'],
    condition_cols=['样本占比'],
    color_cols=['坏样本率', 'LIFT值'],
    condition_color={
        '样本占比': '5B8FF9',
        '坏样本率': ['5AD8A6', 'FFFFCC', 'F76E6C'],
        'LIFT值': ['F76E6C', 'FFFFFF', '5AD8A6'],
    },
    theme_color='fd79a8',
    precision={'坏样本率': 1},
    compact=True,
);

分箱标签,样本总数,样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,累积LIFT值,分档KS值
"[-inf, 0.0478)",194,20.00%,8.8%,-0.5294,0.0462,0.1859,0.62,1.00,0.00
"[0.0478, 0.0719)",194,20.00%,10.3%,-0.3497,0.0215,0.1859,0.74,1.09,0.09
"[0.0719, 0.0984)",194,20.00%,14.4%,0.0338,0.0002,0.1859,1.03,1.21,0.15
"[0.0984, 0.1342)",194,20.00%,12.4%,-0.1442,0.0039,0.1859,0.88,1.31,0.14
"[0.1342, +inf)",194,20.00%,24.2%,0.6733,0.1140,0.1859,1.73,1.73,0.17


## 2. 多级表头、完整 tuple 和复杂值
分箱基准结果含列表，仍可同时配置数值列的数据条与颜色块。

多级表头统一使用 `theme_color` 作为底色；最上层相邻分组之间使用居中的白色短竖线，上下留白，无需额外参数。

In [3]:
benchmark_df = benchmark_binning_methods(
    df, '中智小牛分C3', overdue='MOB1', dpds=[3, 0],
    methods=['quantile', 'cart', 'chi'], max_n_bins=5, n_jobs=1,
)
score_column = next(col for col in benchmark_df.columns if col[-1] == '综合评分')
benchmark_df.show(
    percent_cols=[],
    condition_cols=[score_column],
    color_cols=['最大LIFT值'],
    condition_color={score_column: '00897B', '最大LIFT值': ['FFFFFF', '00897B']},
    theme_color='00695C',
);

## 3. 关闭自动格式
空名单明确关闭百分比和数据条；`fill=False` 关闭隔行底色，保留表头与 hover。

In [4]:
table.show(percent_cols=[], condition_cols=[], color_cols=[], fill=False, compact=True)
pd.testing.assert_frame_equal(table, original)

分箱标签,样本总数,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,累积LIFT值,分档KS值
"[-inf, 0.0478)",194,0.09,-0.5294,0.0462,0.1859,0.62,1.00,0.00
"[0.0478, 0.0719)",194,0.10,-0.3497,0.0215,0.1859,0.74,1.09,0.09
"[0.0719, 0.0984)",194,0.14,0.0338,0.0002,0.1859,1.03,1.21,0.15
"[0.0984, 0.1342)",194,0.12,-0.1442,0.0039,0.1859,0.88,1.31,0.14
"[0.1342, +inf)",194,0.24,0.6733,0.1140,0.1859,1.73,1.73,0.17
